## Importações

In [1]:
import torch
import torchaudio
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoFeatureExtractor, ASTForAudioClassification, TrainingArguments, Trainer, AutoConfig
import evaluate

/Users/sergioo/Documents/GitHub/Deep-Learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("ashraq/esc50")


Repo card metadata block was not found. Setting CardData to empty.


In [3]:
dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
        num_rows: 1600
    })
    test: Dataset({
        features: ['filename', 'fold', 'target', 'category', 'esc10', 'src_file', 'take', 'audio'],
        num_rows: 400
    })
})


In [4]:
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"
feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)

def preprocess_function(exemplos):
    audio_arrays = [x["array"] for x in exemplos["audio"]]
    
    inputs = feature_extractor(
        audio_arrays, 
        sampling_rate=feature_extractor.sampling_rate, 
        padding="max_length",
        max_length=500, 
        truncation=True
    )
    
    return inputs

encoded_dataset = dataset.map(preprocess_function, batched=True)
print("Pré-processamento concluído!")


Pré-processamento concluído!


In [5]:
dataset['train'].features['target']

Value('int64')

In [6]:
pares_unicos = set(zip(dataset["train"]["target"], dataset["train"]["category"]))
pares_ordenados = sorted(list(pares_unicos), key=lambda x: x[0])

# Criamos a lista final só com os nomes
labels = [par[1] for par in pares_ordenados]
num_labels = len(labels)

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

display(pares_ordenados)

[(0, 'dog'),
 (1, 'rooster'),
 (2, 'pig'),
 (3, 'cow'),
 (4, 'frog'),
 (5, 'cat'),
 (6, 'hen'),
 (7, 'insects'),
 (8, 'sheep'),
 (9, 'crow'),
 (10, 'rain'),
 (11, 'sea_waves'),
 (12, 'crackling_fire'),
 (13, 'crickets'),
 (14, 'chirping_birds'),
 (15, 'water_drops'),
 (16, 'wind'),
 (17, 'pouring_water'),
 (18, 'toilet_flush'),
 (19, 'thunderstorm'),
 (20, 'crying_baby'),
 (21, 'sneezing'),
 (22, 'clapping'),
 (23, 'breathing'),
 (24, 'coughing'),
 (25, 'footsteps'),
 (26, 'laughing'),
 (27, 'brushing_teeth'),
 (28, 'snoring'),
 (29, 'drinking_sipping'),
 (30, 'door_wood_knock'),
 (31, 'mouse_click'),
 (32, 'keyboard_typing'),
 (33, 'door_wood_creaks'),
 (34, 'can_opening'),
 (35, 'washing_machine'),
 (36, 'vacuum_cleaner'),
 (37, 'clock_alarm'),
 (38, 'clock_tick'),
 (39, 'glass_breaking'),
 (40, 'helicopter'),
 (41, 'chainsaw'),
 (42, 'siren'),
 (43, 'car_horn'),
 (44, 'engine'),
 (45, 'train'),
 (46, 'church_bells'),
 (47, 'airplane'),
 (48, 'fireworks'),
 (49, 'hand_saw')]

In [7]:
model_id = "MIT/ast-finetuned-audioset-10-10-0.4593"

config = AutoConfig.from_pretrained(model_id)
config.num_labels = num_labels
config.label2id = label2id
config.id2label = id2label


model = ASTForAudioClassification.from_pretrained(
    model_id, 
    config=config,
    ignore_mismatched_sizes=True
)

print("Modelo configurado com sucesso!")

Loading weights: 100%|██████████| 203/203 [00:00<00:00, 6852.56it/s]
[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                          
------------------------+----------+------------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([50, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([50])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Modelo configurado com sucesso!


In [8]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    # Pega as previsões (probabilidades) e descobre qual foi a maior
    predictions = np.argmax(eval_pred.predictions, axis=1)
    # Compara com as respostas corretas
    return accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)

# Renomeando a coluna 'target' para 'labels' para não bugar o modelo
encoded_dataset = encoded_dataset.rename_column("target", "labels")

In [9]:
# Célula 9 - Parâmetros e Início do Treino 
training_args = TrainingArguments(
    output_dir="./resultados_ast_esc50",
    eval_strategy="epoch",        # Avalia a acurácia a cada época
    save_strategy="epoch",        # Salva o progresso a cada época
    learning_rate=3e-5,           # Taxa de aprendizado pequena para o Transfer Learning
    per_device_train_batch_size=4,# Consome menos memória do Mac
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,# Junta passos para não pesar na memória (4x4 = lotes de 16)
    num_train_epochs=3,           # 3 épocas (voltas completas nos dados)
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

print("Iniciando o treinamento! Vá buscar um café ☕...")
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Iniciando o treinamento! Vá buscar um café ☕...


/Users/sergioo/Documents/GitHub/Deep-Learning/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 